In [1]:
!pip install pyspark

In [2]:
from google.colab import files

uploaded = files.upload()

Saving manufacturing_production_120k.csv to manufacturing_production_120k.csv


In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Advanced PySpark Complete Exercises") \
    .getOrCreate()
df = spark.read.csv(
    "manufacturing_production_120k.csv",
    header=True,
    inferSchema=True
)

In [4]:
df.show(20, truncate=False)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|shift  |product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|PR0000001    |2026-08-07     |PL03    |MC011     |OP0275     |Evening|MTR-100     |Motors        |1060         |993           |30             |34              |388.11    |61.4         |3.88          |Rework        |No              |Charan    |
|PR0000002    |2026-

In [5]:
df.printSchema()

root
 |-- production_id: string (nullable = true)
 |-- production_date: date (nullable = true)
 |-- plant_id: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- operator_id: string (nullable = true)
 |-- shift: string (nullable = true)
 |-- product_code: string (nullable = true)
 |-- product_family: string (nullable = true)
 |-- units_planned: integer (nullable = true)
 |-- units_produced: integer (nullable = true)
 |-- defective_units: integer (nullable = true)
 |-- downtime_minutes: integer (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- vibration_mm_s: double (nullable = true)
 |-- quality_status: string (nullable = true)
 |-- maintenance_flag: string (nullable = true)
 |-- supervisor: string (nullable = true)



In [6]:
print("Columns:", df.columns)

Columns: ['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag', 'supervisor']


In [7]:
print("Total Records:", df.count())

Total Records: 120000


In [8]:
df.select("production_id", "plant_id", "machine_id", "product_family", "units_produced").show(10)

+-------------+--------+----------+--------------+--------------+
|production_id|plant_id|machine_id|product_family|units_produced|
+-------------+--------+----------+--------------+--------------+
|    PR0000001|    PL03|     MC011|        Motors|           993|
|    PR0000002|    PL04|     MC019|   Controllers|           836|
|    PR0000003|    PL01|     MC002|        Motors|           380|
|    PR0000004|    PL03|     MC013|         Pumps|           720|
|    PR0000005|    PL04|     MC020|         Pumps|           296|
|    PR0000006|    PL01|     MC002|        Motors|           795|
|    PR0000007|    PL02|     MC007|        Valves|           955|
|    PR0000008|    PL02|     MC009|        Valves|           895|
|    PR0000009|    PL03|     MC015|   Controllers|           289|
|    PR0000010|    PL02|     MC006|   Controllers|           527|
+-------------+--------+----------+--------------+--------------+
only showing top 10 rows


In [9]:
print("Datatypes:", df.dtypes)

Datatypes: [('production_id', 'string'), ('production_date', 'date'), ('plant_id', 'string'), ('machine_id', 'string'), ('operator_id', 'string'), ('shift', 'string'), ('product_code', 'string'), ('product_family', 'string'), ('units_planned', 'int'), ('units_produced', 'int'), ('defective_units', 'int'), ('downtime_minutes', 'int'), ('energy_kwh', 'double'), ('temperature_c', 'double'), ('vibration_mm_s', 'double'), ('quality_status', 'string'), ('maintenance_flag', 'string'), ('supervisor', 'string')]


In [10]:
df.sample(withReplacement=False, fraction=0.001, seed=42).limit(10).show()

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0007283|     2026-03-25|    PL05|     MC025|     OP0320|  Night|     MTR-100|        Motors|          815|           676|             22|              10|    259.99|        47.16|          3.76|          Pass|              No|    Charan|
|    PR0007396|     

In [11]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import lit

plant_data = [
    ("PL01", "Hyderabad Plant", "Hyderabad"),
    ("PL02", "Pune Plant", "Pune"),
    ("PL03", "Chennai Plant", "Chennai")
]

In [12]:
plant_df_list = spark.createDataFrame(plant_data, ["plant_id", "plant_name", "city"])

In [13]:
plant_schema = StructType([
    StructField("plant_id", StringType(), True),
    StructField("plant_name", StringType(), True),
    StructField("city", StringType(), True)
])
plant_df_struct = spark.createDataFrame(plant_data, schema=plant_schema)

In [14]:
plant_df_struct.printSchema()

root
 |-- plant_id: string (nullable = true)
 |-- plant_name: string (nullable = true)
 |-- city: string (nullable = true)



In [15]:
plant_df_country = plant_df_struct.withColumn("country", lit("India"))
plant_df_country.show()

+--------+---------------+---------+-------+
|plant_id|     plant_name|     city|country|
+--------+---------------+---------+-------+
|    PL01|Hyderabad Plant|Hyderabad|  India|
|    PL02|     Pune Plant|     Pune|  India|
|    PL03|  Chennai Plant|  Chennai|  India|
+--------+---------------+---------+-------+



In [17]:
import pandas as pd

pdf = pd.read_csv("manufacturing_production_120k.csv")

In [18]:
print(pdf.head(5))

  production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family  units_planned  units_produced  defective_units  \
0      MTR-100         Motors           1060             993               30   
1      CTL-330    Controllers            809             836                0   
2      MTR-300         Motors            442             380               17   
3      PMP-210          Pumps            787             720                7   
4      PMP-110          Pumps            344             296                7   

   downtime_minutes  energy_kwh  temperature_c  vibratio

In [19]:
print(pdf.tail(5))

       production_id production_date plant_id machine_id operator_id    shift  \
119995     PR0119996      2026-07-13     PL05      MC022      OP0193  Evening   
119996     PR0119997      2026-04-26     PL05      MC024      OP0124  Evening   
119997     PR0119998      2026-02-01     PL06      MC027      OP0051  Evening   
119998     PR0119999      2026-01-17     PL01      MC001      OP0452  Evening   
119999     PR0120000      2026-08-16     PL02      MC007      OP0420  Morning   

       product_code product_family  units_planned  units_produced  \
119995      MTR-200         Motors            435             399   
119996      CTL-130    Controllers            813             777   
119997      VLV-120         Valves            650             597   
119998      MTR-300         Motors            887             899   
119999      MTR-200         Motors            547             477   

        defective_units  downtime_minutes  energy_kwh  temperature_c  \
119995                1   

In [20]:
print("Shape:", pdf.shape)

Shape: (120000, 18)


In [21]:
print("Columns:", pdf.columns.tolist())

Columns: ['production_id', 'production_date', 'plant_id', 'machine_id', 'operator_id', 'shift', 'product_code', 'product_family', 'units_planned', 'units_produced', 'defective_units', 'downtime_minutes', 'energy_kwh', 'temperature_c', 'vibration_mm_s', 'quality_status', 'maintenance_flag', 'supervisor']


In [22]:
print(pdf.dtypes)

production_id        object
production_date      object
plant_id             object
machine_id           object
operator_id          object
shift                object
product_code         object
product_family       object
units_planned         int64
units_produced        int64
defective_units       int64
downtime_minutes    float64
energy_kwh          float64
temperature_c       float64
vibration_mm_s      float64
quality_status       object
maintenance_flag     object
supervisor           object
dtype: object


In [23]:
print(pdf[["plant_id", "machine_id", "units_produced"]].head())

  plant_id machine_id  units_produced
0     PL03      MC011             993
1     PL04      MC019             836
2     PL01      MC002             380
3     PL03      MC013             720
4     PL04      MC020             296


In [24]:
pdf_renamed = pdf.rename(columns={"units_produced": "production_output"})

In [25]:
print("Rejects:", len(pdf[pdf["quality_status"] == "Reject"]))

Rejects: 22744


In [26]:
print("High Temp (>75):", len(pdf[pdf["temperature_c"] > 75]))

High Temp (>75): 6505


In [27]:
print("Night Shift PL01:", len(pdf[(pdf["shift"] == "Night") & (pdf["plant_id"] == "PL01")]))

Night Shift PL01: 6745


In [30]:
import dask.dataframe as dd

# Fix: Explicitly specify the dtype for columns with mixed integers/floats across chunks
ddf = dd.read_csv(
    "manufacturing_production_120k.csv",
    dtype={'downtime_minutes': 'float64'}
)

In [31]:
print(ddf.head())

  production_id production_date plant_id machine_id operator_id    shift  \
0     PR0000001      2026-08-07     PL03      MC011      OP0275  Evening   
1     PR0000002      2026-08-03     PL04      MC019      OP0489  Morning   
2     PR0000003      2026-05-20     PL01      MC002      OP0189  Evening   
3     PR0000004      2026-01-15     PL03      MC013      OP0076  Morning   
4     PR0000005      2026-09-21     PL04      MC020      OP0501  Morning   

  product_code product_family  units_planned  units_produced  defective_units  \
0      MTR-100         Motors           1060             993               30   
1      CTL-330    Controllers            809             836                0   
2      MTR-300         Motors            442             380               17   
3      PMP-210          Pumps            787             720                7   
4      PMP-110          Pumps            344             296                7   

   downtime_minutes  energy_kwh  temperature_c  vibratio

In [32]:
print("Partitions:", ddf.npartitions)

Partitions: 1


In [33]:
ddf_sub = ddf[["plant_id", "product_family", "units_produced"]]

In [34]:
ddf_rejects = ddf[ddf["quality_status"] == "Reject"]

In [35]:
avg_units = ddf.groupby("plant_id")["units_produced"].mean()
total_defects = ddf.groupby("product_family")["defective_units"].sum()

In [36]:
print("Avg Units by Plant:\n", avg_units.compute())
print("Total Defects by Family:\n", total_defects.compute())

Avg Units by Plant:
 plant_id
PL01    672.998304
PL02    672.781326
PL03    675.665503
PL04    673.835101
PL05    673.124994
PL06    678.006884
Name: units_produced, dtype: float64
Total Defects by Family:
 product_family
Controllers    507781
Motors         505287
Pumps          507864
Valves         509312
Name: defective_units, dtype: int64


In [37]:
from pyspark.sql.functions import col, round, lit

df_trans = df \
    .withColumn("production_efficiency", round((col("units_produced") / col("units_planned")) * 100, 2)) \
    .withColumn("good_units", col("units_produced") - col("defective_units")) \
    .withColumn("country", lit("India")) \
    .withColumnRenamed("energy_kwh", "energy_consumed") \
    .withColumn("units_produced", col("units_produced").cast("double")) \
    .withColumn("defect_percentage", round((col("defective_units") / col("units_produced")) * 100, 2)) \
    .withColumn("energy_per_unit", round(col("energy_consumed") / col("units_produced"), 4))

df_trans.select("production_efficiency", "good_units", "country", "defect_percentage", "energy_per_unit").show(5)

+---------------------+----------+-------+-----------------+---------------+
|production_efficiency|good_units|country|defect_percentage|energy_per_unit|
+---------------------+----------+-------+-----------------+---------------+
|                93.68|       963|  India|             3.02|         0.3908|
|               103.34|       836|  India|              0.0|         0.4768|
|                85.97|       363|  India|             4.47|         0.4825|
|                91.49|       713|  India|             0.97|         0.3022|
|                86.05|       289|  India|             2.36|         0.3901|
+---------------------+----------+-------+-----------------+---------------+
only showing top 5 rows


In [38]:
df_drop1 = df.drop("supervisor")
df_drop2 = df.drop("supervisor", "maintenance_flag")

In [39]:
df_no_dups = df.dropDuplicates()
df_unique_prod = df.dropDuplicates(["production_id"])

In [40]:
df_op_clean = df.dropna(subset=["operator_id"])
df_sensor_clean = df.dropna(how="all", subset=["temperature_c", "vibration_mm_s"])

In [41]:
df.select("product_family").distinct().show()
df.select("plant_id", "machine_id").distinct().show(10)

+--------------+
|product_family|
+--------------+
|         Pumps|
|        Valves|
|   Controllers|
|        Motors|
+--------------+

+--------+----------+
|plant_id|machine_id|
+--------+----------+
|    PL01|     MC001|
|    PL01|     MC004|
|    PL04|     MC019|
|    PL04|     MC017|
|    PL05|     MC022|
|    PL04|     MC018|
|    PL01|     MC003|
|    PL06|     MC029|
|    PL02|     MC007|
|    PL03|     MC013|
+--------+----------+
only showing top 10 rows


In [42]:
df.orderBy(col("units_produced").asc()).show(5)
df.orderBy(col("units_produced").desc()).show(5)
df.orderBy(col("defective_units").desc()).show(5)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0038643|     2026-08-06|    PL02|     MC009|     OP0361|Evening|     MTR-100|        Motors|          250|           205|              6|              73|     65.18|        55.99|          2.72|          Pass|              No|    Harini|
|    PR0059668|     

In [43]:
df.orderBy(col("plant_id").asc(), col("units_produced").desc()).show(10)
df.orderBy(col("units_produced").desc()).show(20) # Top 20 runs
df.orderBy(col("downtime_minutes").desc()).show(10) # Highest downtime
df.orderBy(col("vibration_mm_s").desc()).select("machine_id", "plant_id", "vibration_mm_s").show(10)

+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|production_id|production_date|plant_id|machine_id|operator_id|  shift|product_code|product_family|units_planned|units_produced|defective_units|downtime_minutes|energy_kwh|temperature_c|vibration_mm_s|quality_status|maintenance_flag|supervisor|
+-------------+---------------+--------+----------+-----------+-------+------------+--------------+-------------+--------------+---------------+----------------+----------+-------------+--------------+--------------+----------------+----------+
|    PR0073059|     2026-04-01|    PL01|     MC003|     OP0110|Evening|     MTR-200|        Motors|         1198|          1239|             11|               0|     282.6|        50.32|          3.91|        Rework|              No|     Gopal|
|    PR0002678|     

In [44]:
from pyspark.sql.functions import sum, avg, max, min, count

In [45]:
df.select(
    sum("units_planned"), sum("units_produced"), sum("defective_units"),
    round(avg("units_produced"), 2), max("units_produced"), min("units_produced"),
    round(avg("downtime_minutes"), 2), sum("energy_kwh"), round(avg("temperature_c"), 2)
).show()

+------------------+-------------------+--------------------+-----------------------------+-------------------+-------------------+-------------------------------+--------------------+----------------------------+
|sum(units_planned)|sum(units_produced)|sum(defective_units)|round(avg(units_produced), 2)|max(units_produced)|min(units_produced)|round(avg(downtime_minutes), 2)|     sum(energy_kwh)|round(avg(temperature_c), 2)|
+------------------+-------------------+--------------------+-----------------------------+-------------------+-------------------+-------------------------------+--------------------+----------------------------+
|          87094621|           80927888|             2030244|                        674.4|               1245|                205|                          33.93|2.8368014080000117E7|                       63.02|
+------------------+-------------------+--------------------+-----------------------------+-------------------+-------------------+-------------

In [46]:
df.groupBy("plant_id").agg(sum("units_produced").alias("total_production")).show()
df.groupBy("product_family").agg(sum("units_produced").alias("total_output")).orderBy("total_output", ascending=False).show()
df.groupBy("shift").agg(round(avg("downtime_minutes"), 2).alias("avg_downtime")).show()
df.groupBy("quality_status").agg(count("production_id").alias("count")).show()

+--------+----------------+
|plant_id|total_production|
+--------+----------------+
|    PL04|        13529935|
|    PL02|        13546452|
|    PL06|        13492337|
|    PL05|        13323163|
|    PL03|        13547769|
|    PL01|        13488232|
+--------+----------------+

+--------------+------------+
|product_family|total_output|
+--------------+------------+
|        Valves|    20433082|
|         Pumps|    20208739|
|   Controllers|    20168572|
|        Motors|    20117495|
+--------------+------------+

+-------+------------+
|  shift|avg_downtime|
+-------+------------+
|Evening|       34.22|
|Morning|       33.62|
|  Night|       33.95|
+-------+------------+

+--------------+-----+
|quality_status|count|
+--------------+-----+
|        Rework|22928|
|        Reject|22744|
|          Pass|74328|
+--------------+-----+



In [65]:
from google.colab import files

uploaded = files.upload()
from google.colab import files

uploaded = files.upload()

Saving plant_master.csv to plant_master (3).csv


Saving machine_master.csv to machine_master (1).csv


In [66]:
plant_master = spark.read.csv("plant_master.csv", header=True, inferSchema=True)
machine_master = spark.read.csv("machine_master.csv", header=True, inferSchema=True)

In [67]:
prod_plant = df.join(plant_master, "plant_id", "inner")
prod_plant.select("production_id", "plant_name", "units_produced").show(5)

full_df = df.join(plant_master, "plant_id", "left").join(machine_master, ["plant_id", "machine_id"], "left")

+-------------+---------------+--------------+
|production_id|     plant_name|units_produced|
+-------------+---------------+--------------+
|    PR0000001|  Chennai Plant|           993|
|    PR0000002|    Noida Plant|           836|
|    PR0000003|Hyderabad Plant|           380|
|    PR0000004|  Chennai Plant|           720|
|    PR0000005|    Noida Plant|           296|
+-------------+---------------+--------------+
only showing top 5 rows


In [68]:
full_df.groupBy("city").agg(sum("units_produced").alias("total_production")).show()
full_df.groupBy("machine_type").agg(round(avg("downtime_minutes"), 2).alias("avg_downtime")).show()
full_df.groupBy("region").agg(sum("defective_units").alias("total_defects")).show()
full_df.groupBy("plant_name").agg(round(sum("energy_kwh"), 2).alias("total_energy")).show()

+---------+----------------+
|     city|total_production|
+---------+----------------+
|  Chennai|        13547769|
|Ahmedabad|        13323163|
|  Kolkata|        13492337|
|    Noida|        13529935|
|     Pune|        13546452|
|Hyderabad|        13488232|
+---------+----------------+

+------------+------------+
|machine_type|avg_downtime|
+------------+------------+
|         CNC|       33.65|
|     Testing|       33.88|
|     Welding|        34.0|
|    Assembly|       33.97|
|     Casting|       34.17|
+------------+------------+

+------+-------------+
|region|total_defects|
+------+-------------+
| South|       678984|
|  East|       338993|
|  West|       674149|
| North|       338118|
+------+-------------+

+---------------+------------+
|     plant_name|total_energy|
+---------------+------------+
|    Noida Plant|  4746238.58|
|  Kolkata Plant|  4723180.16|
|  Chennai Plant|  4742748.17|
|     Pune Plant|  4757984.95|
|Hyderabad Plant|  4725471.19|
|Ahmedabad Plant|  4672

In [69]:
from pyspark.sql.functions import when

df_u1 = df.withColumn("quality_status", when(col("quality_status") == "Reject", "Failed").otherwise(col("quality_status"))) \
          .withColumn("units_planned", when(col("product_family") == "Motors", round(col("units_planned") * 1.05)).otherwise(col("units_planned"))) \
          .withColumn("operator_id", when(col("operator_id").isNull(), "UNASSIGNED").otherwise(col("operator_id"))) \
          .withColumn("supervisor", when(col("supervisor") == "Anita", "Anita Rao").otherwise(col("supervisor")))

df_u1.select("quality_status", "product_family", "operator_id", "supervisor").show(5)

+--------------+--------------+-----------+----------+
|quality_status|product_family|operator_id|supervisor|
+--------------+--------------+-----------+----------+
|        Rework|        Motors|     OP0275|    Charan|
|        Failed|   Controllers|     OP0489|     Farah|
|        Failed|        Motors|     OP0189|     Deepa|
|        Rework|         Pumps|     OP0076|     Farah|
|          Pass|         Pumps|     OP0501|     Deepa|
+--------------+--------------+-----------+----------+
only showing top 5 rows


In [70]:
print("Missing Operators:", df.filter(col("operator_id").isNull()).count())
print("Missing Temp:", df.filter(col("temperature_c").isNull()).count())
print("Missing Vibration:", df.filter(col("vibration_mm_s").isNull()).count())

Missing Operators: 1524
Missing Temp: 2132
Missing Vibration: 1788


In [71]:
df_dropped = df.na.drop()
df_filled = df.fillna({"downtime_minutes": 0, "operator_id": "UNKNOWN"})
avg_temp = df.select(avg("temperature_c")).first()[0]
df_imputed = df.fillna({"temperature_c": avg_temp})

In [72]:
from pyspark.sql.functions import upper, lower, trim, length, concat_ws, abs, regexp_replace, substring, coalesce

df_fn = df \
    .withColumn("fam_upper", upper(col("product_family"))) \
    .withColumn("sup_lower", lower(col("supervisor"))) \
    .withColumn("op_trim", trim(col("operator_id"))) \
    .withColumn("code_len", length(col("product_code"))) \
    .withColumn("plant_machine", concat_ws("-", col("plant_id"), col("machine_id"))) \
    .withColumn("diff_units", abs(col("units_planned") - col("units_produced"))) \
    .withColumn("clean_id", regexp_replace(col("production_id"), "PR", "ID_"))

df_fn.select("fam_upper", "sup_lower", "plant_machine", "diff_units", "clean_id").show(5)

+-----------+---------+-------------+----------+----------+
|  fam_upper|sup_lower|plant_machine|diff_units|  clean_id|
+-----------+---------+-------------+----------+----------+
|     MOTORS|   charan|   PL03-MC011|        67|ID_0000001|
|CONTROLLERS|    farah|   PL04-MC019|        27|ID_0000002|
|     MOTORS|    deepa|   PL01-MC002|        62|ID_0000003|
|      PUMPS|    farah|   PL03-MC013|        67|ID_0000004|
|      PUMPS|    deepa|   PL04-MC020|        48|ID_0000005|
+-----------+---------+-------------+----------+----------+
only showing top 5 rows


In [73]:
df_cat = df.withColumn("eff_pct", (col("units_produced") / col("units_planned")) * 100) \
    .withColumn("Efficiency_Category",
        when(col("eff_pct") >= 100, "Above Target")
        .when(col("eff_pct") >= 95, "On Target")
        .when(col("eff_pct") >= 85, "Below Target")
        .otherwise("Critical")
    ) \
    .withColumn("Defect_Category",
        when(col("defective_units") == 0, "Perfect")
        .when(col("defective_units").between(1, 10), "Normal")
        .otherwise("High Defect")
    )

df_cat.groupBy("Efficiency_Category", "Defect_Category").count().show()

+-------------------+---------------+-----+
|Efficiency_Category|Defect_Category|count|
+-------------------+---------------+-----+
|       Below Target|         Normal|16455|
|       Above Target|         Normal| 5455|
|          On Target|        Perfect| 2591|
|           Critical|    High Defect| 9650|
|       Above Target|    High Defect|14238|
|           Critical|        Perfect| 1636|
|          On Target|         Normal| 7261|
|           Critical|         Normal| 5535|
|          On Target|    High Defect|16973|
|       Above Target|        Perfect| 2042|
|       Below Target|        Perfect| 5222|
|       Below Target|    High Defect|32942|
+-------------------+---------------+-----+



In [74]:
from google.colab import files

uploaded = files.upload()

Saving machine_configurations.json to machine_configurations (1).json


In [75]:
from pyspark.sql.functions import explode

machine_json = spark.read.option("multiline", "true").json("machine_configurations.json")

In [76]:
json_ext = machine_json.select(
    col("machine_id"), col("plant_id"), col("machine_type"),
    col("specifications.rated_power_kw").alias("rated_power_kw"),
    col("specifications.max_temperature_c").alias("max_temperature_c")
)

In [77]:
sensors_exploded = machine_json.select(
    col("machine_id"), explode(col("sensors")).alias("sensor")
).select("machine_id", col("sensor.sensor").alias("sensor_type"), col("sensor.active").alias("active"))

sensors_exploded.filter((col("sensor_type") == "temperature") & (col("active") == True)).show(5)

+----------+-----------+------+
|machine_id|sensor_type|active|
+----------+-----------+------+
|     MC001|temperature|  true|
|     MC002|temperature|  true|
|     MC003|temperature|  true|
|     MC004|temperature|  true|
|     MC005|temperature|  true|
+----------+-----------+------+
only showing top 5 rows


In [78]:
from google.colab import files

uploaded = files.upload()

Saving manufacturing_production_extra_12k.csv to manufacturing_production_extra_12k (2).csv


In [79]:
extra_df = spark.read.csv("manufacturing_production_extra_12k.csv", header=True, inferSchema=True)

In [93]:
count_main = df.count()
print("Main DataFrame record count:", count_main)

Main DataFrame record count: 120000


In [94]:
count_extra = extra_df.count()
print("Extra DataFrame record count:", count_extra)

Extra DataFrame record count: 12500


In [95]:
unioned_df = df.union(extra_df)

In [96]:
union_all_df = df.unionAll(extra_df)

In [98]:
print("Union count:", unioned_df.count())
print("UnionAll count:", union_all_df.count())

Union count: 132500
UnionAll count: 132500


In [99]:
duplicates_df = unioned_df.groupBy("production_id").count().filter(col("count") > 1)
print("Duplicate production_id records found:")
duplicates_df.show(5)

Duplicate production_id records found:
+-------------+-----+
|production_id|count|
+-------------+-----+
|    PR0000413|    2|
|    PR0000135|    2|
|    PR0000236|    2|
|    PR0000339|    2|
|    PR0000059|    2|
+-------------+-----+
only showing top 5 rows


In [100]:
deduped_df = unioned_df.dropDuplicates(["production_id"])

In [102]:
print("Count before dropDuplicates:", unioned_df.count())
print("   Count after dropDuplicates:", deduped_df.count())

Count before dropDuplicates: 132500
   Count after dropDuplicates: 132000


In [103]:
union_by_name_df = df.unionByName(extra_df)
print("UnionByName count:", union_by_name_df.count())

UnionByName count: 132500


In [106]:
extra_reordered = extra_df.select(list(reversed(extra_df.columns)))
union_reordered = df.unionByName(extra_reordered)
print("Reordered UnionByName count:", union_reordered.count())

Reordered UnionByName count: 132500


In [107]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, lag, lead, sum as spark_sum

window_spec = Window.partitionBy("plant_id").orderBy(col("units_produced").desc())
seq_window = Window.partitionBy("plant_id", "machine_id").orderBy("production_date")

df_window = df \
    .withColumn("rank", rank().over(window_spec)) \
    .withColumn("row_num", row_number().over(window_spec)) \
    .withColumn("cum_units", spark_sum("units_produced").over(seq_window)) \
    .withColumn("prev_units", lag("units_produced", 1).over(seq_window))

df_window.select("plant_id", "machine_id", "units_produced", "rank", "cum_units", "prev_units").show(10)

+--------+----------+--------------+----+---------+----------+
|plant_id|machine_id|units_produced|rank|cum_units|prev_units|
+--------+----------+--------------+----+---------+----------+
|    PL01|     MC001|          1157| 310|    12621|      NULL|
|    PL01|     MC001|          1078|1104|    12621|      1157|
|    PL01|     MC001|          1027|1945|    12621|      1078|
|    PL01|     MC001|          1023|2012|    12621|      1027|
|    PL01|     MC001|           997|2495|    12621|      1023|
|    PL01|     MC001|           929|4029|    12621|       997|
|    PL01|     MC001|           926|4100|    12621|       929|
|    PL01|     MC001|           822|6560|    12621|       926|
|    PL01|     MC001|           817|6689|    12621|       822|
|    PL01|     MC001|           799|7083|    12621|       817|
+--------+----------+--------------+----+---------+----------+
only showing top 10 rows


In [108]:
from pyspark.sql.functions import to_date, year, month, dayofmonth, date_format, date_add, datediff, current_date

df_date = df.withColumn("prod_date", to_date(col("production_date"), "yyyy-MM-dd")) \
    .withColumn("prod_year", year("prod_date")) \
    .withColumn("prod_month", month("prod_date")) \
    .withColumn("year_month", date_format("prod_date", "yyyy-MM")) \
    .withColumn("plus_7_days", date_add(col("prod_date"), 7)) \
    .withColumn("days_from_today", datediff(current_date(), col("prod_date")))

df_date.groupBy("year_month").agg(spark_sum("units_produced").alias("monthly_units")).orderBy("year_month").show()

spark.stop()

+----------+-------------+
|year_month|monthly_units|
+----------+-------------+
|   2026-01|      8478059|
|   2026-02|      7810891|
|   2026-03|      8441121|
|   2026-04|      8211005|
|   2026-05|      8507542|
|   2026-06|      8400505|
|   2026-07|      8476416|
|   2026-08|      8529150|
|   2026-09|      8115221|
|   2026-10|      5957978|
+----------+-------------+

